In [ ]:
from ultralytics import YOLO
# from ultralytics import settings
# settings['tensorboard'] = True
# settings._save()

model = YOLO("models/base/yolo26s-obb.pt") 

In [ ]:
import albumentations as A

IMG_SIZE = 720

# Aggressive augmentations layered on top of Ultralytics' built-in geometric/color jitter.
# Targets robustness to phone-camera artefacts: tight/partial crops, sensor noise, JPEG compression.
augmentations = [
    # A.RandomSizedBBoxSafeCrop(height=IMG_SIZE, width=IMG_SIZE, erosion_rate=0.2, p=0.3),
    # A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(0.05, 0.2), hole_width_range=(0.05, 0.2), p=0.3),
    A.ISONoise(p=0.2),
    A.ImageCompression(quality_range=(40, 90), p=0.3),
    A.Blur(blur_limit=5, p=0.1),
    A.MedianBlur(blur_limit=5, p=0.05),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomGamma(p=0.2),
    A.ToGray(p=0.02),
    A.CLAHE(p=0.05),
]

# Shared with the augmentation-preview cell below so the preview matches training exactly.
train_kwargs = dict(
    data="./datasets/SKU110K_fixed/sku110k_obb.yaml",
    epochs=150,
    imgsz=IMG_SIZE,
    single_cls=True,
    degrees=10.0,    # Slightly rotate images
    flipud=0.5,      # Flip up-down
    fliplr=0.5,      # Flip left-righ
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    batch=8,
    augmentations=augmentations,
    device=0,
)

In [ ]:
# Preview the *exact* training-time pipeline (mosaic, affine/rotate/shear/perspective,
# mixup, the custom Albumentations transforms, HSV jitter, flips) on real batches,
# without launching a full training run.
#
# Note: the first run will scan/cache all labels in datasets/SKU110K_fixed/labels/train
# (one-time cost, also speeds up the real training run afterwards).
import matplotlib.pyplot as plt
from ultralytics.cfg import DEFAULT_CFG, get_cfg
from ultralytics.data.build import build_dataloader, build_yolo_dataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.plotting import plot_images

preview_batch = 4  # smaller than train batch so each image is easier to inspect
preview_cfg = get_cfg(DEFAULT_CFG, {**train_kwargs, "task": "obb", "mode": "train"})
data = check_det_dataset(preview_cfg.data)

dataset = build_yolo_dataset(preview_cfg, img_path=data["train"], batch=preview_batch, data=data, mode="train")
loader = build_dataloader(dataset, batch=preview_batch, workers=0, shuffle=True)

for i, batch in zip(range(3), loader):
    grid = plot_images(batch, names=data["names"], save=False, threaded=False)
    plt.figure(figsize=(10, 10))
    plt.imshow(grid)
    plt.axis("off")
    plt.title(f"augmented batch {i}")
    plt.show()

In [ ]:
model.train(**train_kwargs)

     23/150      5.98G       1.06     0.5543    0.01502    0.01163       1098        736: 100% ━━━━━━━━━━━━ 7192/7192 4.6it/s 26:180.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 515/515 9.8it/s 52.4s<0.2ss
                   all       4116     636776      0.913      0.874      0.918        0.7

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss angle_loss  Instances       Size
     24/150      7.47G      1.058     0.5494     0.0148    0.01165       1410        736: 100% ━━━━━━━━━━━━ 7192/7192 4.8it/s 25:040.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 515/515 10.1it/s 50.8s0.2ss
                   all       4116     636776      0.913      0.874      0.918        0.7

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss angle_loss  Instances       Size
     25/150      7.47G      1.061     0.5502    0.01457    0.01187       2503        736: 20% ━━

In [1]:
model.save("./models/tuned/items_detect.yolo26n-obb.pt")

NameError: name 'model' is not defined

In [ ]:
import glob
import os
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO("./models/tuned/items_detect.yolo26n-obb.pt")

print("loaded model")
test_images_path = "./datasets/SKU110K_fixed/images/test/*.jpg"  # Update path if needed
image_paths = glob.glob(test_images_path)

if not image_paths:
    raise FileNotFoundError("No images found. Please verify your test images path.")

# Select a random subset to display (e.g., 4 samples)
num_samples = min(4, len(image_paths))
sampled_paths = random.sample(image_paths, num_samples)

# 3. Setup Matplotlib grid (assuming a 2x2 grid for 4 samples)
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.ravel()  # Flatten the 2D array of axes into 1D for easy iteration

# 4. Infer and plot
for idx, img_path in enumerate(sampled_paths):
    # Run segmentation prediction
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    
    # Extract the first result from the list
    result = results[0]
    

    annotated_img = result.plot(pil=True, boxes=True, masks=True)
    
    axes[idx].imshow(annotated_img)
    axes[idx].set_title(os.path.basename(img_path), fontsize=10)
    axes[idx].axis("off")  # Hide grid borders and coordinates

# Tight layout adjustments for notebook embedding
plt.tight_layout()
plt.show()